In [ ]:
import json
import sys
import uuid
from itertools import product
from pathlib import Path

import hiplot as hip
import numpy as np
import pandas as pd
from IPython.display import Javascript, display

project_root = Path.cwd()
if not (project_root / "tasks").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from network_creation.plotting.reference_values import get_observer_reference_style
from tasks.policies import load_policy_results

In [ ]:
def filter_frame(frame, filters):
    """Return a filtered copy of ``frame`` while preserving all columns."""
    filtered = frame.copy()
    for column, raw_values in filters.items():
        if column not in filtered.columns:
            available = ", ".join(map(str, filtered.columns))
            raise KeyError(
                f"unknown filter column {column!r}; available columns: {available}"
            )

        if isinstance(
            raw_values, (list, tuple, set, np.ndarray, pd.Index)
        ) and not isinstance(raw_values, (str, bytes)):
            values = list(raw_values)
        else:
            values = [raw_values]

        if not values:
            raise ValueError(f"filter for {column!r} must contain at least one value")

        series = filtered[column]
        if pd.api.types.is_numeric_dtype(series) and all(
            isinstance(value, (int, float, np.integer, np.floating)) for value in values
        ):
            numeric = series.to_numpy(dtype=float)
            mask = np.zeros(len(filtered), dtype=bool)
            for value in values:
                if np.isnan(float(value)):
                    mask |= np.isnan(numeric)
                else:
                    mask |= np.isclose(numeric, float(value), rtol=0.0, atol=1e-9)
            filtered = filtered.loc[mask]
        else:
            filtered = filtered.loc[series.isin(values)]

    if filtered.empty:
        raise ValueError("reference_filter removed all rows")
    return filtered.reset_index(drop=True)


def _finite_unique_values(frame, column):
    if column not in frame.columns:
        raise ValueError(
            f"target references require column {column!r}; keep it in parallel_frame"
        )

    values = pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=float)
    finite_values = values[np.isfinite(values)]
    if finite_values.size == 0:
        raise ValueError(f"target references require finite {column!r} values")

    return np.unique(np.round(finite_values, 12))


def _reference_context_columns(observable):
    columns = ["n_agents"]
    if observable == "centered_mean_welfare.mean":
        columns.extend(["benefit_to_cost_ratio", "distance_discount"])
    return columns


def _context_mask(frame, context):
    mask = np.ones(len(frame), dtype=bool)
    for column, value in context.items():
        values = pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=float)
        mask &= np.isfinite(values) & np.isclose(values, value, rtol=0.0, atol=1e-9)
    return mask


def _reference_contexts(frame, observable):
    columns = _reference_context_columns(observable)
    unique_values = {column: _finite_unique_values(frame, column) for column in columns}
    label_columns = [
        column for column, values in unique_values.items() if len(values) > 1
    ]

    for values in product(*(unique_values[column] for column in columns)):
        context = dict(zip(columns, values, strict=True))
        context_frame = frame.loc[_context_mask(frame, context)]
        if not context_frame.empty:
            yield context, label_columns, context_frame


def _format_context_value(value):
    value = float(value)
    rounded = round(value)
    if np.isclose(value, rounded, rtol=0.0, atol=1e-9):
        return str(int(rounded))
    return f"{value:.6g}"


def _context_label(column):
    return {
        "benefit_to_cost_ratio": "b/c",
        "distance_discount": "discount",
    }.get(column, column)


def _contextual_reference_label(label, context, label_columns):
    if not label_columns:
        return str(label)
    context_text = ", ".join(
        f"{_context_label(column)}={_format_context_value(context[column])}"
        for column in label_columns
    )
    return f"{context_text}: {label}"


def _context_n_agents(context):
    value = float(context["n_agents"])
    rounded = round(value)
    if not np.isclose(value, rounded, rtol=0.0, atol=1e-9):
        raise ValueError(f"n_agents reference context must be integral; got {value}")
    return int(rounded)


def _reference_context_payload(context):
    return {
        str(column): float(value)
        for column, value in context.items()
        if np.isfinite(float(value))
    }


def target_references_for_frame(frame, observable, theta_labels):
    """Resolve observer reference values for the displayed HiPlot frame."""
    if "target" not in frame.columns:
        raise ValueError("target_references_for_frame requires a 'target' column")

    missing_theta = [label for label in theta_labels if label not in frame.columns]
    if missing_theta:
        missing = ", ".join(missing_theta)
        raise ValueError(
            f"target_references_for_frame is missing theta columns: {missing}"
        )

    labeled_references = {}
    labeled_contexts = {}
    for context, label_columns, context_frame in _reference_contexts(frame, observable):
        references, _reference_styles = get_observer_reference_style(
            observable,
            context_frame["target"].to_numpy(dtype=float),
            n_agents=_context_n_agents(context),
            theta_labels=theta_labels,
            theta_values=context_frame.loc[:, theta_labels].to_numpy(dtype=float),
        )
        for label, value in references.items():
            if np.isfinite(float(value)):
                reference_label = _contextual_reference_label(
                    label, context, label_columns
                )
                labeled_references[reference_label] = float(value)
                labeled_contexts[reference_label] = _reference_context_payload(context)
    target_references_for_frame.last_contexts = labeled_contexts
    return labeled_references


def display_hiplot_with_target_references(
    hip_exp,
    references,
    target_column="target",
    target_reference_contexts=None,
    value_format="{:.2f}",
    right_padding=180,
):
    """Display a HiPlot experiment and overlay labeled reference arrows on target."""
    hip_display = hip_exp.display()
    if target_reference_contexts is None:
        target_reference_contexts = getattr(
            target_references_for_frame, "last_contexts", {}
        )
    reference_items = [
        {
            "label": str(label),
            "value": float(value),
            "text": f"{label} ({value_format.format(float(value))})",
            "context": target_reference_contexts.get(str(label), {}),
        }
        for label, value in references.items()
        if np.isfinite(float(value))
    ]
    hip_exp._target_reference_overlay_js = ""
    if not reference_items:
        return hip_display

    overlay_class = f"hiplot-target-reference-overlay-{uuid.uuid4().hex}"
    marker_id = f"{overlay_class}-marker"
    overlay_js = f"""
(() => {{
  const references = {json.dumps(reference_items)};
  const targetColumn = {json.dumps(str(target_column))};
  const sharedOverlayClass = "hiplot-target-reference-overlay";
  const overlayClass = {json.dumps(overlay_class)};
  const markerId = {json.dumps(marker_id)};
  const rightPadding = Number({json.dumps(float(right_padding))});
  const ns = "http://www.w3.org/2000/svg";
  let animationFrame = null;
  let drawTimer = null;
  let lastDrawSignature = "";
  let observer = null;

  function parseTranslate(transform) {{
    const match = /translate\\(\\s*([-+0-9.eE]+)(?:[,\\s]+([-+0-9.eE]+))?\\s*\\)/.exec(transform || "");
    if (!match) {{
      return [0, 0];
    }}
    return [Number(match[1] || 0), Number(match[2] || 0)];
  }}

  function makeSvg(tag, attrs) {{
    const node = document.createElementNS(ns, tag);
    for (const [key, value] of Object.entries(attrs || {{}})) {{
      node.setAttribute(key, value);
    }}
    return node;
  }}

  function observeRoot() {{
    if (!observer) {{
      return;
    }}
    document.querySelectorAll(".pplot-root").forEach((root) => {{
      observer.observe(root, {{ childList: true, subtree: true, attributes: true }});
    }});
  }}

  function ensureRightPadding(svg) {{
    const padding = Number.isFinite(rightPadding) ? Math.max(0, rightPadding) : 0;
    if (padding <= 0) {{
      return;
    }}
    if (!svg.dataset.targetReferenceBaseWidth) {{
      const attrWidth = Number(svg.getAttribute("width"));
      const rectWidth = svg.getBoundingClientRect().width;
      const baseWidth = Number.isFinite(attrWidth) && attrWidth > 0 ? attrWidth : rectWidth;
      svg.dataset.targetReferenceBaseWidth = String(baseWidth);
      const viewBox = svg.getAttribute("viewBox");
      if (viewBox) {{
        svg.dataset.targetReferenceBaseViewBox = viewBox;
      }}
    }}

    const baseWidth = Number(svg.dataset.targetReferenceBaseWidth);
    const paddedWidth = baseWidth + padding;
    svg.setAttribute("width", String(paddedWidth));
    svg.style.width = `${{paddedWidth}}px`;
    svg.style.maxWidth = "none";

    const baseViewBox = svg.dataset.targetReferenceBaseViewBox;
    if (baseViewBox) {{
      const parts = baseViewBox.trim().split(/[\\s,]+/).map(Number);
      if (parts.length === 4 && parts.every(Number.isFinite)) {{
        parts[2] = parts[2] + padding;
        svg.setAttribute("viewBox", parts.join(" "));
      }}
    }}

    const root = svg.closest(".pplot-root");
    if (root) {{
      root.style.paddingRight = `${{padding}}px`;
      root.style.boxSizing = "content-box";
    }}
    if (svg.parentElement) {{
      svg.parentElement.style.paddingRight = `${{padding}}px`;
      svg.parentElement.style.boxSizing = "content-box";
    }}
  }}

  function axisLabel(axis) {{
    const label = axis.querySelector(".label-name");
    return label ? label.textContent.trim() : "";
  }}

  function findAxis(svg, column) {{
    return Array.from(svg.querySelectorAll("g.dimension")).find((axis) => {{
      return axisLabel(axis) === column;
    }});
  }}

  function findTargetAxis(svg) {{
    return findAxis(svg, targetColumn);
  }}

  function axisTicks(axis) {{
    return Array.from(axis.querySelectorAll("g.tick"))
      .map((tick) => {{
        const text = tick.querySelector("text");
        if (!text) {{
          return null;
        }}
        const value = Number(text.textContent.trim().replace(/,/g, ""));
        const y = parseTranslate(tick.getAttribute("transform"))[1];
        if (!Number.isFinite(value) || !Number.isFinite(y)) {{
          return null;
        }}
        return {{ value, y }};
      }})
      .filter(Boolean);
  }}

  function interpolateY(value, ticks) {{
    if (ticks.length < 2) {{
      return null;
    }}
    const sorted = ticks.slice().sort((a, b) => a.value - b.value);
    const low = sorted[0];
    const high = sorted[sorted.length - 1];
    if (value < low.value || value > high.value || Math.abs(high.value - low.value) < 1e-12) {{
      return null;
    }}
    const ratio = (value - low.value) / (high.value - low.value);
    return low.y + ratio * (high.y - low.y);
  }}

  function valueAtY(y, ticks) {{
    if (ticks.length < 2) {{
      return null;
    }}
    const sorted = ticks.slice().sort((a, b) => a.y - b.y);
    const minY = sorted[0].y;
    const maxY = sorted[sorted.length - 1].y;
    if (y < minY - 1e-6 || y > maxY + 1e-6) {{
      return null;
    }}
    for (let index = 1; index < sorted.length; index += 1) {{
      const low = sorted[index - 1];
      const high = sorted[index];
      if (low.y <= y && y <= high.y && Math.abs(high.y - low.y) >= 1e-12) {{
        const ratio = (y - low.y) / (high.y - low.y);
        return low.value + ratio * (high.value - low.value);
      }}
    }}
    const edge = Math.abs(y - minY) < Math.abs(y - maxY) ? sorted[0] : sorted[sorted.length - 1];
    return edge.value;
  }}

  function brushPixels(axis) {{
    const brush = axis.querySelector(".pplot-brush") || Array.from(axis.querySelectorAll("g")).find((node) => {{
      return node.__brush && node.__brush.selection;
    }});
    const selection = brush && brush.__brush ? brush.__brush.selection : null;
    if (!selection) {{
      return null;
    }}
    if (Array.isArray(selection[0])) {{
      return [Number(selection[0][1]), Number(selection[1][1])];
    }}
    return [Number(selection[0]), Number(selection[1])];
  }}

  function activeBrushRanges(svg) {{
    const ranges = {{}};
    for (const axis of Array.from(svg.querySelectorAll("g.dimension"))) {{
      const column = axisLabel(axis);
      const pixels = brushPixels(axis);
      if (!column || !pixels) {{
        continue;
      }}
      const ticks = axisTicks(axis);
      const values = pixels.map((pixel) => valueAtY(pixel, ticks));
      if (values.every(Number.isFinite)) {{
        ranges[column] = [Math.min(values[0], values[1]), Math.max(values[0], values[1])];
      }}
    }}
    return ranges;
  }}

  function currentSelectedRows() {{
    const instance = window.hiplot_last_instance;
    const rows = instance && instance.state && (instance.state.rows_selected || instance.state.rows_filtered);
    return Array.isArray(rows) ? rows : [];
  }}

  function nearlyEqual(left, right) {{
    return Number.isFinite(left) && Number.isFinite(right) && Math.abs(left - right) <= 1e-9;
  }}

  function inRange(value, range) {{
    return Number.isFinite(value) && value >= range[0] - 1e-9 && value <= range[1] + 1e-9;
  }}

  function referenceMatchesSelectedRows(item, rows) {{
    const entries = Object.entries(item.context || {{}});
    if (!entries.length || !rows.length) {{
      return true;
    }}
    return rows.some((row) => entries.every(([column, value]) => {{
      return nearlyEqual(Number(row[column]), Number(value));
    }}));
  }}

  function referenceMatchesBrushes(item, brushRanges) {{
    if (brushRanges[targetColumn] && !inRange(Number(item.value), brushRanges[targetColumn])) {{
      return false;
    }}
    for (const [column, value] of Object.entries(item.context || {{}})) {{
      if (brushRanges[column] && !inRange(Number(value), brushRanges[column])) {{
        return false;
      }}
    }}
    return true;
  }}

  function clearOverlay(svg) {{
    svg.querySelectorAll(`.${{sharedOverlayClass}}`).forEach((node) => node.remove());
  }}

  function drawOverlay() {{
    if (observer) {{
      observer.disconnect();
    }}
    try {{
      const svgs = Array.from(document.querySelectorAll(".pplot-root svg"));
      const svg = svgs[svgs.length - 1];
      if (!svg) {{
        return false;
      }}

      const targetAxis = findTargetAxis(svg);
      if (!targetAxis) {{
        return false;
      }}

      const ticks = axisTicks(targetAxis);
      if (ticks.length < 2) {{
        return false;
      }}
      const brushRanges = activeBrushRanges(svg);
      const selectedRows = currentSelectedRows();
      const visibleReferences = references.filter((item) => {{
        return referenceMatchesBrushes(item, brushRanges) && referenceMatchesSelectedRows(item, selectedRows);
      }});
      const signature = JSON.stringify({{
        targetTransform: targetAxis.getAttribute("transform") || "",
        ticks: ticks.map((tick) => [tick.value, tick.y]),
        brushRanges,
        visibleReferences: visibleReferences.map((item) => [item.label, item.value]),
      }});
      const overlayPresent = Boolean(svg.querySelector(`g.${{overlayClass}}`));
      if (signature === lastDrawSignature && overlayPresent) {{
        return true;
      }}
      lastDrawSignature = signature;

      clearOverlay(svg);

      svg.style.overflow = "visible";
      const root = svg.closest(".pplot-root");
      if (root) {{
        root.style.overflow = "visible";
      }}
      if (svg.parentElement) {{
        svg.parentElement.style.overflow = "visible";
      }}
      ensureRightPadding(svg);

      const lineStart = 18;
      const lineEnd = 3;
      const textX = 22;
      const anchor = "start";

      const defs = makeSvg("defs", {{ class: `${{sharedOverlayClass}} ${{overlayClass}}` }});
      const marker = makeSvg("marker", {{
        id: markerId,
        markerWidth: "7",
        markerHeight: "7",
        refX: "6",
        refY: "3.5",
        orient: "auto",
        markerUnits: "strokeWidth",
        viewBox: "0 0 7 7",
      }});
      marker.appendChild(makeSvg("path", {{ d: "M 0 0 L 7 3.5 L 0 7 z", fill: "#c03728" }}));
      defs.appendChild(marker);
      svg.insertBefore(defs, svg.firstChild);

      const layer = makeSvg("g", {{ class: `${{sharedOverlayClass}} ${{overlayClass}}`, "pointer-events": "none" }});
      targetAxis.appendChild(layer);

      for (const item of visibleReferences) {{
        const y = interpolateY(item.value, ticks);
        if (y === null) {{
          continue;
        }}
        const group = makeSvg("g", {{ class: "target-reference" }});
        group.appendChild(makeSvg("line", {{
          x1: String(lineStart),
          x2: String(lineEnd),
          y1: String(y),
          y2: String(y),
          stroke: "#c03728",
          "stroke-width": "1.6",
          "marker-end": `url(#${{markerId}})`,
        }}));

        const label = makeSvg("text", {{
          x: String(textX),
          y: String(y),
          dy: "0.32em",
          fill: "#c03728",
          "font-family": "sans-serif",
          "font-size": "11",
          "font-weight": "600",
          "paint-order": "stroke",
          stroke: "white",
          "stroke-width": "3",
          "stroke-linejoin": "round",
          "text-anchor": anchor,
        }});
        label.textContent = item.text;
        group.appendChild(label);
        layer.appendChild(group);
      }}
      return true;
    }} finally {{
      observeRoot();
    }}
  }}

  function scheduleDraw(mutations) {{
    if (Array.isArray(mutations) && mutations.length && mutations.every((mutation) => {{
      const target = mutation.target;
      return target && target.nodeType === 1 && target.closest(`.${{sharedOverlayClass}}`);
    }})) {{
      return;
    }}
    if (drawTimer) {{
      window.clearTimeout(drawTimer);
    }}
    if (animationFrame) {{
      window.cancelAnimationFrame(animationFrame);
    }}
    drawTimer = window.setTimeout(() => {{
      drawTimer = null;
      animationFrame = window.requestAnimationFrame(drawOverlay);
    }}, 80);
  }}

  let attempts = 0;
  const interval = window.setInterval(() => {{
    attempts += 1;
    if (drawOverlay() || attempts > 80) {{
      window.clearInterval(interval);
    }}
  }}, 100);

  observer = new MutationObserver(scheduleDraw);
  observeRoot();
  window.addEventListener("resize", scheduleDraw);
}})();
"""
    hip_exp._target_reference_overlay_js = overlay_js
    display(Javascript(overlay_js))
    return hip_display

In [ ]:
zarr_path = project_root / "artifacts" / "simulation_runs" / "run-20260521T234239.zarr"
observable = "centered_mean_welfare.mean"
normalize_time = True

In [ ]:
ds = load_policy_results(zarr_path, normalize_time=normalize_time, stats=[observable])
time_mode = "normalized" if normalize_time else "raw"
time_axis = ds.attrs.get("time_axis", "raw_simulator_steps")
final_step = float(ds.coords["step"].values[-1])

print("time mode:", time_mode)
print("time axis:", time_axis)
print("final step:", final_step)

theta_labels = [str(value) for value in ds.coords["theta_parameter"].values]
sweep_dims = tuple(dim for dim in ds["theta"].dims if dim != "theta_parameter")
sweep_columns = [
    dim for dim in sweep_dims if dim != "parameter_row" and dim not in theta_labels
]

theta = ds["theta"].transpose(*sweep_dims, "theta_parameter").to_numpy()
target = (
    ds["x"].sel(observable=observable).isel(step=-1).transpose(*sweep_dims).to_numpy()
)

mask = np.isfinite(theta).all(axis=-1) & np.isfinite(target)
theta_subset = theta[mask]
target_subset = target[mask]

parallel_frame = pd.DataFrame(theta_subset, columns=theta_labels)
for dim, values in zip(
    sweep_dims,
    np.meshgrid(
        *(ds.coords[dim].to_numpy() for dim in sweep_dims),
        indexing="ij",
    ),
    strict=True,
):
    if dim in sweep_columns:
        parallel_frame[dim] = values[mask]
parallel_frame["target"] = target_subset

dimensions = [*sweep_columns, *theta_labels, "target"]
parallel_frame = parallel_frame.loc[:, dimensions]

reference_filter = {"n_agents": 100}
hiplot_frame = filter_frame(parallel_frame, reference_filter)

print("loaded:", zarr_path.relative_to(project_root))
print("rows:", len(parallel_frame))
print("display rows:", len(hiplot_frame))
print("reference filter:", reference_filter)
print("sweep dims:", list(sweep_dims))
print("dimensions:", dimensions)
hiplot_frame.head()

In [ ]:
hip_exp = hip.Experiment.from_dataframe(hiplot_frame)
hip_exp.colorby = "target"

target_references = target_references_for_frame(hiplot_frame, observable, theta_labels)
target_reference_contexts = getattr(target_references_for_frame, "last_contexts", {})
if target_references:
    target_range_values = np.array(
        [*hiplot_frame["target"].to_numpy(dtype=float), *target_references.values()],
        dtype=float,
    )
    target_range_values = target_range_values[np.isfinite(target_range_values)]
    hip_exp.parameters_definition["target"].force_range(
        float(target_range_values.min()),
        float(target_range_values.max()),
    )

display_hiplot_with_target_references(
    hip_exp,
    target_references,
    target_reference_contexts=target_reference_contexts,
)

In [ ]:
if False:
    export_path = project_root / "artifacts" / "hiplot_with_references.html"
    export_path.parent.mkdir(parents=True, exist_ok=True)

    overlay_js = getattr(hip_exp, "_target_reference_overlay_js", "")
    if target_references and not overlay_js:
        raise RuntimeError(
            "Run the HiPlot display cell above before exporting so the reference overlay is generated."
        )

    html = hip_exp.to_html()
    if overlay_js:
        overlay_script = f"<script>{overlay_js}</script>"
        if "</body>" in html:
            html = html.replace("</body>", f"{overlay_script}</body>")
        else:
            html = f"{html}\n{overlay_script}"

    export_path.write_text(html, encoding="utf-8")
    export_path.relative_to(project_root)